Undersampling + Sliding Window

In [3]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
train_df = pd.concat([train_df, valid_df], ignore_index=True)
train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        

        classes = np.unique(window_labels_step1)
        
        for c in classes:

            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            

            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                c_indices = np.random.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [2, 6]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            

        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):

        actual_idx = self.valid_indices[idx]
        

        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
MAX_SAMPLES = 25000 
TIME_STEPS = 10
BATCH_SIZE = 256


TRAIN_STEP_SIZE = 5 
TEST_STEP_SIZE = 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=10000000, step=1)
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 5)...
Đang tính toán các cửa sổ hợp lệ (global step=5) và Undersampling...
Class 0: Giữ nguyên 15874 cửa sổ (step=5).
Class 1: Giảm từ 387254 xuống 25000 cửa sổ.
Class 2: Giữ nguyên 2009 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giảm từ 29000 xuống 25000 cửa sổ.
Class 4: Giữ nguyên 15179 cửa sổ (step=5).
Class 5: Giữ nguyên 1958 cửa sổ (step=5).
Class 6: Giữ nguyên 9474 cửa sổ (sử dụng step=1 để bảo toàn).
Class 7: Giảm từ 85945 xuống 25000 cửa sổ.
Class 8: Giữ nguyên 13395 cửa sổ (step=5).
Class 9: Giảm từ 33215 xuống 25000 cửa sổ.
Class 10: Giữ nguyên 14850 cửa sổ (step=5).
Tổng số cửa sổ sau khi lọc và Undersampling: 172739
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 79368 cửa sổ (step=1).
Class 1: Giữ nguyên 1936269 cửa sổ (step=1).
Class 2: Giữ nguyên 10048 cửa sổ (sử dụng step=1 để bảo toàn).
C

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# hàm tính focal loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]


class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

#squeze and excitation block: chấm điểm độ quan trọng từng kênh đặc trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) 
        y = self.fc(y).view(b, c, 1)    
        return x * y.expand_as(x)       

# residual bao gồm cnn + SE Block
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        

        self.se = SEBlock1D(out_channels)
        

        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out



class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
   
        x = x.permute(0, 2, 1) 
        

        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        

        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {DEVICE}")
# khởi tạo mô hình
model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
print(model)

Đang sử dụng thiết bị: cuda
CNN_BiLSTM_Attention(
  (res1): ResidualBlock1D(
    (conv1): Conv1d(314, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (relu): ReLU()
    (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (dropout): Dropout1d(p=0.2, inplace=False)
    (se): SEBlock1D(
      (avg_pool): AdaptiveAvgPool1d(output_size=1)
      (fc): Sequential(
        (0): Linear(in_features=64, out_features=8, bias=False)
        (1): ReLU(inplace=True)
        (2): Linear(in_features=8, out_features=64, bias=False)
        (3): Sigmoid()
      )
    )
    (shortcut): Sequential(
      (0): Conv1d(314, 64, kernel_size=(1,), stride=(1,))
      (1): GroupNorm(8, 64, eps=1e-05, affine=True)
    )
  )
  (res2): ResidualBlock1D(
    (conv1): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (relu):

In [9]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

# tính lại phân phối nhãn (labels) thực tế sau khi đã áp dụng Undersampling trong tập train
actual_labels = []
for idx in train_dataset.valid_indices:
    label = train_dataset.y[idx + TIME_STEPS - 1].item()
    actual_labels.append(label)

actual_labels = np.array(actual_labels)

classes_in_train = np.unique(actual_labels)
new_weights = compute_class_weight(
    class_weight='balanced', 
    classes=classes_in_train, 
    y=actual_labels
)

# căn bậc 2 trọng số
smoothed_new_weights = np.sqrt(new_weights)

# khởi tạo focal loss với trọng số đã được căn bậc 2
alpha_tensor = torch.tensor(smoothed_new_weights, dtype=torch.float32).to(DEVICE)
# Tăng Gamma lên 3.0 để model bị ép phải tập trung mạnh hơn vào các mẫu KHÓ (do Concept Drift)
criterion = FocalLoss(alpha=alpha_tensor, gamma=3.0)

print(f"Trọng số mới sau khi Undersampling và Smooth:\n{alpha_tensor.cpu().numpy()}")

Trọng số mới sau khi Undersampling và Smooth:
[0.9946165  0.79255396 2.7958179  0.79255396 1.0171319  2.8319952
 1.2874553  0.79255396 1.0827483  0.79255396 1.0283374 ]


In [10]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

# Dùng Cross Entropy với Label Smoothing = 0.1 để chống over-confidence thay vì Focal Loss (hạn chế kìm nghẹt F1 các class có Concept Drift)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Dùng Adam với weight decay nhỉnh hơn xíu (1e-3)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate nếu val loss không cải thiện sau 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 5

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # Áp dụng Gradient Clipping để cắt tỉa các vi phân nảy loạn xạ khi model học các mẫu Hard bị drift
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.9160, Train Macro F1: 0.8351 | Val Loss: 0.5513, Val Macro F1: 0.9343


Epoch [2/30] - Train Loss: 0.7182, Train Macro F1: 0.9385 | Val Loss: 0.5397, Val Macro F1: 0.9560


Epoch [3/30] - Train Loss: 0.6989, Train Macro F1: 0.9491 | Val Loss: 0.5404, Val Macro F1: 0.9590


Epoch [4/30] - Train Loss: 0.6825, Train Macro F1: 0.9559 | Val Loss: 0.5488, Val Macro F1: 0.9335


...

Epoch [27/30] - Train Loss: 0.6054, Train Macro F1: 0.9915 | Val Loss: 0.5198, Val Macro F1: 0.9903


Epoch [28/30] - Train Loss: 0.6034, Train Macro F1: 0.9925 | Val Loss: 0.5184, Val Macro F1: 0.9933


Epoch [29/30] - Train Loss: 0.6025, Train Macro F1: 0.9923 | Val Loss: 0.5195, Val Macro F1: 0.9916


Epoch [30/30] - Train Loss: 0.6019, Train Macro F1: 0.9929 | Val Loss: 0.5216, Val Macro F1: 0.9864

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.89      0.82      0.86     19846
           1       1.00      1.00      1.00    484077
           2       0.46      1.00      0.63      2515
           3       1.00      0.84      0.91     36253
           4       0.92      0.87      0.90     18979
           5       1.00      1.00      1.00      2451
           6       0.95      0.77      0.85     11847
           7       1.00      1.00      1.00    107436
           8       0.73      0.99      0.84     16746
           9       0.94      0.98      0.96     41514
          10       1.00      0.99      0.99     18567

    accuracy                           0.98    760231
   macro avg       0.90      0.93      0.90    760231
weighted avg       0.98      0.98      0.98    760231



In [11]:
print(classification_report(all_test_targets, all_test_preds, digits=4))

              precision    recall  f1-score   support

           0     0.8935    0.8225    0.8565     19846
           1     1.0000    1.0000    1.0000    484077
           2     0.4586    0.9980    0.6284      2515
           3     0.9998    0.8383    0.9120     36253
           4     0.9182    0.8745    0.8959     18979
           5     0.9984    0.9976    0.9980      2451
           6     0.9525    0.7675    0.8500     11847
           7     0.9983    1.0000    0.9991    107436
           8     0.7254    0.9919    0.8380     16746
           9     0.9440    0.9765    0.9600     41514
          10     0.9953    0.9898    0.9925     18567

    accuracy                         0.9792    760231
   macro avg     0.8985    0.9324    0.9028    760231
weighted avg     0.9831    0.9792    0.9798    760231

